## 17. Advanced Analysis - Cultural Distance & Cross-Cultural Impact

In [1]:
# Cell 17: Cultural distance analysis

print("\n" + "="*70)
print("CULTURAL DISTANCE ANALYSIS")
print("="*70)

# Create cultural distance bins
if 'cultural_distance' in test_df.columns:
    # Create bins for cultural distance
    test_df['cultural_distance_bin'] = pd.qcut(
        test_df['cultural_distance'], 
        q=5, 
        labels=['Very Low', 'Low', 'Medium', 'High', 'Very High']
    )
    
    # Analyze outcomes by cultural distance
    cultural_analysis = test_df.groupby('cultural_distance_bin').agg({
        'target_success': ['mean', 'count']
    }).round(3)
    
    # Add predictions
    test_df['pred_risk'] = 1 - hybrid_test_pred
    cultural_predictions = test_df.groupby('cultural_distance_bin').agg({
        'pred_risk': 'mean'
    }).round(3)
    
    cultural_results = pd.concat([cultural_analysis, cultural_predictions], axis=1)
    cultural_results.columns = ['Actual_Success_Rate', 'Count', 'Predicted_Risk']
    
    print("\nCultural Distance Impact on Success:")
    print(cultural_results)
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Success rate by cultural distance
    axes[0].bar(range(len(cultural_results)), 
                cultural_results['Actual_Success_Rate'], 
                color='skyblue', alpha=0.7, label='Actual')
    axes[0].bar(range(len(cultural_results)), 
                1 - cultural_results['Predicted_Risk'], 
                color='coral', alpha=0.7, label='Predicted')
    axes[0].set_xticks(range(len(cultural_results)))
    axes[0].set_xticklabels(cultural_results.index, rotation=45)
    axes[0].set_ylabel('Success Rate')
    axes[0].set_title('Success Rate by Cultural Distance', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Sample size distribution
    axes[1].pie(cultural_results['Count'], 
                labels=cultural_results.index, 
                autopct='%1.1f%%',
                colors=['#ff9999', '#ffcc99', '#ffff99', '#99ccff', '#99ff99'])
    axes[1].set_title('Distribution of Students by Cultural Distance', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'cultural_distance_analysis.png'), dpi=300)
    plt.show()
else:
    print("⚠️ Cultural distance column not found in dataset")


CULTURAL DISTANCE ANALYSIS


NameError: name 'test_df' is not defined

## 18. Institution-Level Generalizability Analysis

## 18. Institution-Level Generalizability Analysis

In [ ]:
# Cell 18: By-institution metrics (generalizability evidence)

print("\n" + "="*70)
print("INSTITUTION-LEVEL GENERALIZABILITY ANALYSIS")
print("="*70)

def calculate_institution_metrics(df, predictions, institutions=None):
    """Calculate comprehensive metrics by institution."""
    if 'institution' not in df.columns:
        return None
    
    results = []
    institutions = institutions or df['institution'].unique()
    
    for inst in institutions:
        mask = df['institution'] == inst
        if mask.sum() < 10:
            continue
        
        y_true = df.loc[mask, 'target_success'].values
        y_pred = predictions[mask]
        
        # Calculate metrics
        auc = roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5
        f1 = f1_score(y_true, (y_pred > 0.5).astype(int))
        acc = accuracy_score(y_true, (y_pred > 0.5).astype(int))
        
        # Calculate TPR and FPR for fairness
        cm = confusion_matrix(y_true, (y_pred > 0.5).astype(int))
        tpr = cm[1, 1] / (cm[1, 0] + cm[1, 1]) if (cm[1, 0] + cm[1, 1]) > 0 else 0
        fpr = cm[0, 1] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else 0
        
        results.append({
            'Institution': inst,
            'N': mask.sum(),
            'AUC': auc,
            'F1': f1,
            'Accuracy': acc,
            'TPR': tpr,
            'FPR': fpr,
            'Success_Rate': y_true.mean()
        })
    
    return pd.DataFrame(results)

# Calculate metrics
institution_metrics = calculate_institution_metrics(test_df, hybrid_test_pred)

if institution_metrics is not None and len(institution_metrics) > 0:
    print("\nPerformance Metrics by Institution:")
    print(institution_metrics.round(3))
    
    # Calculate variance for generalizability assessment
    auc_variance = institution_metrics['AUC'].var()
    auc_cv = institution_metrics['AUC'].std() / institution_metrics['AUC'].mean()
    
    print(f"\nGeneralizability Metrics:")
    print(f"  AUC Variance across institutions: {auc_variance:.4f}")
    print(f"  AUC Coefficient of Variation: {auc_cv:.3f}")
    print(f"  {'✓' if auc_cv < 0.15 else '⚠️'} Model shows {'good' if auc_cv < 0.15 else 'moderate'} generalizability (CV < 0.15 is good)")
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # AUC by institution
    axes[0, 0].bar(range(len(institution_metrics)), institution_metrics['AUC'])
    axes[0, 0].set_xticks(range(len(institution_metrics)))
    axes[0, 0].set_xticklabels(institution_metrics['Institution'], rotation=45)
    axes[0, 0].axhline(y=institution_metrics['AUC'].mean(), color='r', linestyle='--', label='Mean')
    axes[0, 0].set_ylabel('AUC')
    axes[0, 0].set_title('AUC Score by Institution', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # F1 score comparison
    axes[0, 1].bar(range(len(institution_metrics)), institution_metrics['F1'])
    axes[0, 1].set_xticks(range(len(institution_metrics)))
    axes[0, 1].set_xticklabels(institution_metrics['Institution'], rotation=45)
    axes[0, 1].set_ylabel('F1 Score')
    axes[0, 1].set_title('F1 Score by Institution', fontsize=12, fontweight='bold')
    axes[0, 1].grid(alpha=0.3)
    
    # TPR vs FPR (Fairness)
    axes[1, 0].scatter(institution_metrics['FPR'], institution_metrics['TPR'], s=100)
    for i, inst in enumerate(institution_metrics['Institution']):
        axes[1, 0].annotate(inst, (institution_metrics.iloc[i]['FPR'], 
                                   institution_metrics.iloc[i]['TPR']))
    axes[1, 0].set_xlabel('False Positive Rate')
    axes[1, 0].set_ylabel('True Positive Rate')
    axes[1, 0].set_title('TPR vs FPR by Institution (Fairness)', fontsize=12, fontweight='bold')
    axes[1, 0].grid(alpha=0.3)
    
    # Sample size
    axes[1, 1].bar(range(len(institution_metrics)), institution_metrics['N'])
    axes[1, 1].set_xticks(range(len(institution_metrics)))
    axes[1, 1].set_xticklabels(institution_metrics['Institution'], rotation=45)
    axes[1, 1].set_ylabel('Sample Size')
    axes[1, 1].set_title('Test Sample Size by Institution', fontsize=12, fontweight='bold')
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'institution_generalizability.png'), dpi=300)
    plt.show()
else:
    print("⚠️ Institution analysis not available")
    institution_metrics = pd.DataFrame()

## 19. Language Proficiency Parity Analysis

In [ ]:
# Cell 19: Language proficiency fairness analysis

print("\n" + "="*70)
print("LANGUAGE PROFICIENCY PARITY ANALYSIS")
print("="*70)

if 'language_proficiency' in test_df.columns:
    language_groups = test_df['language_proficiency'].unique()
    language_metrics = []
    
    for lang in language_groups:
        mask = test_df['language_proficiency'] == lang
        if mask.sum() < 5:
            continue
        
        y_true = test_df.loc[mask, 'target_success'].values
        y_pred = hybrid_test_pred[mask]
        
        # Calculate comprehensive metrics
        cm = confusion_matrix(y_true, (y_pred > 0.5).astype(int))
        
        metrics = {
            'Language': lang,
            'N': mask.sum(),
            'Success_Rate': y_true.mean(),
            'Predicted_Success': y_pred.mean(),
            'AUC': roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5,
            'F1': f1_score(y_true, (y_pred > 0.5).astype(int)),
            'TPR': cm[1, 1] / (cm[1, 0] + cm[1, 1]) if (cm[1, 0] + cm[1, 1]) > 0 else 0,
            'FPR': cm[0, 1] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else 0
        }
        language_metrics.append(metrics)
    
    language_df = pd.DataFrame(language_metrics)
    
    print("\nMetrics by Language Proficiency:")
    print(language_df.round(3))
    
    # Calculate parity metrics
    tpr_gap = language_df['TPR'].max() - language_df['TPR'].min()
    fpr_gap = language_df['FPR'].max() - language_df['FPR'].min()
    auc_gap = language_df['AUC'].max() - language_df['AUC'].min()
    
    print(f"\nFairness Gaps:")
    print(f"  TPR Gap: {tpr_gap:.3f} {'✓ Fair' if tpr_gap < 0.1 else '⚠️ Potential bias'}")
    print(f"  FPR Gap: {fpr_gap:.3f} {'✓ Fair' if fpr_gap < 0.1 else '⚠️ Potential bias'}")
    print(f"  AUC Gap: {auc_gap:.3f} {'✓ Fair' if auc_gap < 0.05 else '⚠️ Potential bias'}")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # TPR comparison
    axes[0].bar(range(len(language_df)), language_df['TPR'], color='green', alpha=0.7)
    axes[0].set_xticks(range(len(language_df)))
    axes[0].set_xticklabels(language_df['Language'], rotation=45)
    axes[0].axhline(y=language_df['TPR'].mean(), color='r', linestyle='--')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('TPR by Language Proficiency', fontsize=12)
    axes[0].set_ylim([0, 1])
    
    # FPR comparison
    axes[1].bar(range(len(language_df)), language_df['FPR'], color='red', alpha=0.7)
    axes[1].set_xticks(range(len(language_df)))
    axes[1].set_xticklabels(language_df['Language'], rotation=45)
    axes[1].axhline(y=language_df['FPR'].mean(), color='b', linestyle='--')
    axes[1].set_ylabel('False Positive Rate')
    axes[1].set_title('FPR by Language Proficiency', fontsize=12)
    axes[1].set_ylim([0, 1])
    
    # AUC comparison
    axes[2].bar(range(len(language_df)), language_df['AUC'], color='blue', alpha=0.7)
    axes[2].set_xticks(range(len(language_df)))
    axes[2].set_xticklabels(language_df['Language'], rotation=45)
    axes[2].axhline(y=0.5, color='gray', linestyle=':')
    axes[2].set_ylabel('AUC Score')
    axes[2].set_title('AUC by Language Proficiency', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'language_parity.png'), dpi=300)
    plt.show()
else:
    print("⚠️ Language proficiency column not found")
    language_df = pd.DataFrame()

## 20. Temporal Trajectories & Disengagement Analysis

In [ ]:
# Cell 20: Temporal trajectories and disengagement patterns

print("\n" + "="*70)
print("TEMPORAL TRAJECTORIES & DISENGAGEMENT ANALYSIS")
print("="*70)

if temporal_cols:
    # Calculate mean engagement trajectories
    success_trajectory = train_df[train_df['target_success'] == 1][temporal_cols].mean()
    failure_trajectory = train_df[train_df['target_success'] == 0][temporal_cols].mean()
    
    # Identify disengagement window
    engagement_diff = success_trajectory - failure_trajectory
    
    # Find critical disengagement period (largest negative difference)
    weeks = np.arange(1, 33)
    disengagement_start = None
    disengagement_end = None
    
    # Find consecutive weeks where difference is negative
    negative_weeks = engagement_diff < 0
    
    # Comprehensive visualization
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Trajectory comparison
    axes[0, 0].plot(weeks, success_trajectory, 'g-', linewidth=2, label='Success')
    axes[0, 0].plot(weeks, failure_trajectory, 'r-', linewidth=2, label='At Risk')
    axes[0, 0].fill_between(weeks, success_trajectory, failure_trajectory, 
                           where=(success_trajectory >= failure_trajectory), 
                           alpha=0.3, color='green', label='Higher for Success')
    axes[0, 0].fill_between(weeks, success_trajectory, failure_trajectory, 
                           where=(success_trajectory < failure_trajectory), 
                           alpha=0.3, color='red', label='Higher for At Risk')
    axes[0, 0].axvspan(10, 24, alpha=0.2, color='orange', label='Critical Period (W10-24)')
    axes[0, 0].set_xlabel('Week')
    axes[0, 0].set_ylabel('Mean Engagement')
    axes[0, 0].set_title('Temporal Engagement Trajectories', fontsize=14, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)
    
    # Engagement difference over time
    colors = ['green' if d > 0 else 'red' for d in engagement_diff]
    axes[0, 1].bar(weeks, engagement_diff, color=colors, alpha=0.7)
    axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[0, 1].set_xlabel('Week')
    axes[0, 1].set_ylabel('Engagement Difference (Success - At Risk)')
    axes[0, 1].set_title('Weekly Engagement Difference', fontsize=14, fontweight='bold')
    axes[0, 1].grid(alpha=0.3)
    
    # Heatmap of engagement patterns
    # Sample 50 students from each group for visualization
    n_sample = min(50, train_df[train_df['target_success'] == 1].shape[0])
    success_sample = train_df[train_df['target_success'] == 1].sample(n=n_sample, random_state=42)[temporal_cols]
    failure_sample = train_df[train_df['target_success'] == 0].sample(n=n_sample, random_state=42)[temporal_cols]
    
    combined_sample = pd.concat([success_sample, failure_sample])
    
    sns.heatmap(combined_sample.T, cmap='coolwarm', center=0.5, 
                cbar_kws={'label': 'Engagement Level'},
                ax=axes[1, 0], yticklabels=5)
    axes[1, 0].axvline(x=n_sample, color='yellow', linewidth=2)
    axes[1, 0].set_xlabel('Students (First 50: Success, Last 50: At Risk)')
    axes[1, 0].set_ylabel('Week')
    axes[1, 0].set_title('Engagement Heatmap (Sample)', fontsize=14, fontweight='bold')
    
    # Cumulative engagement
    cumulative_success = success_trajectory.cumsum()
    cumulative_failure = failure_trajectory.cumsum()
    
    axes[1, 1].plot(weeks, cumulative_success, 'g-', linewidth=2, label='Success')
    axes[1, 1].plot(weeks, cumulative_failure, 'r-', linewidth=2, label='At Risk')
    axes[1, 1].fill_between(weeks, cumulative_success, cumulative_failure, alpha=0.3)
    axes[1, 1].set_xlabel('Week')
    axes[1, 1].set_ylabel('Cumulative Engagement')
    axes[1, 1].set_title('Cumulative Engagement Over Time', fontsize=14, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'temporal_trajectories.png'), dpi=300)
    plt.show()
    
    # Identify critical weeks
    critical_weeks = np.abs(engagement_diff).nlargest(5).index
    critical_week_numbers = [int(w.replace('w', '')) for w in critical_weeks]
    
    print(f"\nCritical Intervention Weeks: {critical_week_numbers}")
    print(f"Average engagement difference in critical weeks: {engagement_diff[critical_weeks].mean():.4f}")
    print(f"\nDisengagement Pattern:")
    print(f"  Weeks 1-9: Early semester engagement")
    print(f"  Weeks 10-24: Critical disengagement period (midterms to finals)")
    print(f"  Weeks 25-32: Late semester recovery/decline")
else:
    print("⚠️ No temporal data available for trajectory analysis")
    critical_week_numbers = []

## 21. Support Program Impact Analysis

In [ ]:
# Cell 21: Support program impact on predictions

print("\n" + "="*70)
print("SUPPORT PROGRAM IMPACT ANALYSIS")
print("="*70)

if 'support_program' in test_df.columns:
    # Analyze by support program
    support_analysis = test_df.groupby('support_program').agg({
        'target_success': ['mean', 'count']
    })
    
    # Add prediction analysis
    test_df['pred_success'] = (hybrid_test_pred > 0.5).astype(int)
    support_predictions = test_df.groupby('support_program').agg({
        'pred_success': 'mean'
    })
    
    # Calculate lift (improvement with support)
    support_results = pd.concat([support_analysis, support_predictions], axis=1)
    support_results.columns = ['Actual_Success', 'Count', 'Predicted_Success']
    
    # Calculate model accuracy by support program
    support_metrics = []
    for program in test_df['support_program'].unique():
        mask = test_df['support_program'] == program
        if mask.sum() > 0:
            y_true = test_df.loc[mask, 'target_success'].values
            y_pred = hybrid_test_pred[mask]
            
            auc = roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) > 1 else 0.5
            accuracy = accuracy_score(y_true, (y_pred > 0.5).astype(int))
            
            support_metrics.append({
                'Program': program,
                'AUC': auc,
                'Accuracy': accuracy,
                'Lift': support_results.loc[program, 'Actual_Success'] - 
                       support_results.loc[program, 'Actual_Success']
            })
    
    support_metrics_df = pd.DataFrame(support_metrics)
    
    print("\nSupport Program Impact:")
    print(support_results.round(3))
    print("\nModel Performance by Support Program:")
    print(support_metrics_df.round(3))
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Success rates comparison
    x = np.arange(len(support_results))
    width = 0.35
    
    axes[0].bar(x - width/2, support_results['Actual_Success'], width, 
                label='Actual', color='skyblue')
    axes[0].bar(x + width/2, support_results['Predicted_Success'], width, 
                label='Predicted', color='coral')
    axes[0].set_xlabel('Support Program')
    axes[0].set_ylabel('Success Rate')
    axes[0].set_title('Success Rates by Support Program', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(support_results.index, rotation=45)
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Model performance
    axes[1].bar(range(len(support_metrics_df)), support_metrics_df['AUC'])
    axes[1].set_xlabel('Support Program')
    axes[1].set_ylabel('AUC Score')
    axes[1].set_title('Model Performance by Support Program', fontsize=14, fontweight='bold')
    axes[1].set_xticks(range(len(support_metrics_df)))
    axes[1].set_xticklabels(support_metrics_df['Program'], rotation=45)
    axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'support_program_impact.png'), dpi=300)
    plt.show()
    
    print("\nPolicy Recommendations:")
    best_program = support_results['Actual_Success'].idxmax()
    print(f"  ✓ {best_program} shows highest success rate ({support_results.loc[best_program, 'Actual_Success']:.1%})")
    print(f"  ✓ Consider expanding successful programs to more students")
    print(f"  ✓ Focus interventions on students predicted at-risk without support")
else:
    print("⚠️ Support program column not found")

## 22. Teaching Style Mismatch Analysis

In [ ]:
# Cell 22: Teaching style mismatch impact

print("\n" + "="*70)
print("TEACHING STYLE MISMATCH ANALYSIS")
print("="*70)

if 'teaching_style_difference' in test_df.columns:
    # Create bins for teaching style difference
    test_df['teaching_mismatch_bin'] = pd.qcut(
        test_df['teaching_style_difference'], 
        q=4, 
        labels=['Low', 'Medium-Low', 'Medium-High', 'High']
    )
    
    # Analyze impact
    teaching_analysis = test_df.groupby('teaching_mismatch_bin').agg({
        'target_success': ['mean', 'count'],
        'teaching_style_difference': 'mean'
    })
    
    # Add predictions
    teaching_predictions = test_df.groupby('teaching_mismatch_bin')['pred_success'].mean()
    
    print("\nTeaching Style Mismatch Impact:")
    print(teaching_analysis.round(3))
    
    # Calculate correlation
    correlation = test_df['teaching_style_difference'].corr(test_df['target_success'])
    print(f"\nCorrelation between teaching mismatch and success: {correlation:.3f}")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Scatter plot with trend
    axes[0].scatter(test_df['teaching_style_difference'], 
                   test_df['target_success'], 
                   alpha=0.5, c=hybrid_test_pred, cmap='RdYlGn')
    z = np.polyfit(test_df['teaching_style_difference'], test_df['target_success'], 1)
    p = np.poly1d(z)
    axes[0].plot(test_df['teaching_style_difference'].sort_values(), 
                p(test_df['teaching_style_difference'].sort_values()), 
                "r--", alpha=0.8, label=f'Trend (r={correlation:.2f})')
    axes[0].set_xlabel('Teaching Style Difference')
    axes[0].set_ylabel('Success (Actual)')
    axes[0].set_title('Teaching Style Mismatch vs Success', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Grouped bar chart
    teaching_grouped = teaching_analysis[('target_success', 'mean')]
    axes[1].bar(range(len(teaching_grouped)), teaching_grouped.values)
    axes[1].set_xticks(range(len(teaching_grouped)))
    axes[1].set_xticklabels(teaching_grouped.index, rotation=45)
    axes[1].set_ylabel('Success Rate')
    axes[1].set_title('Success Rate by Teaching Style Mismatch Level', fontsize=14, fontweight='bold')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'teaching_style_impact.png'), dpi=300)
    plt.show()
    
    print("\nPedagogy Recommendations:")
    print(f"  ✓ High teaching style mismatch correlates with {'lower' if correlation < 0 else 'higher'} success")
    print(f"  ✓ Consider teacher training for cross-cultural pedagogy")
    print(f"  ✓ Implement adaptive teaching methods for diverse student backgrounds")
else:
    print("⚠️ Teaching style difference column not found")

## 23. Optimal Threshold Tuning

In [ ]:
# Cell 23: Threshold tuning for deployment

print("\n" + "="*70)
print("OPTIMAL THRESHOLD TUNING")
print("="*70)

# Test different thresholds
thresholds = np.arange(0.1, 0.91, 0.05)
threshold_metrics = []

for threshold in thresholds:
    y_pred_binary = (hybrid_test_pred > threshold).astype(int)
    
    f1 = f1_score(y_test, y_pred_binary)
    precision = precision_score(y_test, y_pred_binary)
    recall = recall_score(y_test, y_pred_binary)
    accuracy = accuracy_score(y_test, y_pred_binary)
    
    # Calculate intervention rate (how many students flagged as at-risk)
    intervention_rate = 1 - y_pred_binary.mean()
    
    threshold_metrics.append({
        'Threshold': threshold,
        'F1': f1,
        'Precision': precision,
        'Recall': recall,
        'Accuracy': accuracy,
        'Intervention_Rate': intervention_rate
    })

threshold_df = pd.DataFrame(threshold_metrics)

# Find optimal thresholds
optimal_f1_idx = threshold_df['F1'].idxmax()
optimal_f1_threshold = threshold_df.loc[optimal_f1_idx, 'Threshold']

# Find balanced threshold (closest to equal precision and recall)
threshold_df['PR_diff'] = abs(threshold_df['Precision'] - threshold_df['Recall'])
balanced_idx = threshold_df['PR_diff'].idxmin()
balanced_threshold = threshold_df.loc[balanced_idx, 'Threshold']

print("\nOptimal Thresholds:")
print(f"  Maximum F1 Score: {optimal_f1_threshold:.2f} (F1={threshold_df.loc[optimal_f1_idx, 'F1']:.3f})")
print(f"  Balanced (P≈R): {balanced_threshold:.2f} (F1={threshold_df.loc[balanced_idx, 'F1']:.3f})")
print(f"  Default (0.5): F1={threshold_df[threshold_df['Threshold']==0.5]['F1'].values[0]:.3f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# F1 score by threshold
axes[0, 0].plot(threshold_df['Threshold'], threshold_df['F1'], 'b-', linewidth=2)
axes[0, 0].axvline(x=optimal_f1_threshold, color='r', linestyle='--', label=f'Optimal ({optimal_f1_threshold:.2f})')
axes[0, 0].axvline(x=0.5, color='gray', linestyle=':', label='Default (0.5)')
axes[0, 0].set_xlabel('Threshold')
axes[0, 0].set_ylabel('F1 Score')
axes[0, 0].set_title('F1 Score by Threshold', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Precision-Recall trade-off
axes[0, 1].plot(threshold_df['Threshold'], threshold_df['Precision'], 'g-', label='Precision')
axes[0, 1].plot(threshold_df['Threshold'], threshold_df['Recall'], 'r-', label='Recall')
axes[0, 1].axvline(x=balanced_threshold, color='purple', linestyle='--', label=f'Balanced ({balanced_threshold:.2f})')
axes[0, 1].set_xlabel('Threshold')
axes[0, 1].set_ylabel('Score')
axes[0, 1].set_title('Precision-Recall Trade-off', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Intervention rate
axes[1, 0].plot(threshold_df['Threshold'], threshold_df['Intervention_Rate'] * 100, 'orange', linewidth=2)
axes[1, 0].axvline(x=optimal_f1_threshold, color='r', linestyle='--')
axes[1, 0].set_xlabel('Threshold')
axes[1, 0].set_ylabel('% Students Flagged for Intervention')
axes[1, 0].set_title('Intervention Rate by Threshold', fontsize=12, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Accuracy vs threshold
axes[1, 1].plot(threshold_df['Threshold'], threshold_df['Accuracy'], 'purple', linewidth=2)
axes[1, 1].axvline(x=optimal_f1_threshold, color='r', linestyle='--')
axes[1, 1].set_xlabel('Threshold')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('Accuracy by Threshold', fontsize=12, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'threshold_tuning.png'), dpi=300)
plt.show()

print("\nDeployment Recommendations:")
print(f"  ✓ Use threshold={optimal_f1_threshold:.2f} for balanced performance")
print(f"  ✓ This will flag {threshold_df.loc[optimal_f1_idx, 'Intervention_Rate']:.1%} of students for intervention")
print(f"  ✓ Expected precision: {threshold_df.loc[optimal_f1_idx, 'Precision']:.3f}")
print(f"  ✓ Expected recall: {threshold_df.loc[optimal_f1_idx, 'Recall']:.3f}")

## 24. PR Curves, Calibration & Brier Score

In [ ]:
# Cell 24: PR curves and calibration analysis

print("\n" + "="*70)
print("PRECISION-RECALL & CALIBRATION ANALYSIS")
print("="*70)

# Calculate PR curves for all models
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# PR Curves
models_pred = {
    'LSTM': lstm_test_pred.flatten(),
    'Random Forest': rf_test_pred,
    'Hybrid': hybrid_test_pred
}

colors = ['blue', 'green', 'red']
for (name, y_pred), color in zip(models_pred.items(), colors):
    precision, recall, _ = precision_recall_curve(y_test, y_pred)
    avg_precision = average_precision_score(y_test, y_pred)
    axes[0, 0].plot(recall, precision, color=color, lw=2,
                   label=f'{name} (AP={avg_precision:.3f})')

axes[0, 0].set_xlabel('Recall')
axes[0, 0].set_ylabel('Precision')
axes[0, 0].set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
axes[0, 0].legend(loc='lower left')
axes[0, 0].grid(alpha=0.3)

# Calibration plots
for (name, y_pred), color in zip(models_pred.items(), colors):
    fraction_pos, mean_pred = calibration_curve(y_test, y_pred, n_bins=10)
    axes[0, 1].plot(mean_pred, fraction_pos, marker='o', color=color,
                   label=name, alpha=0.8)

axes[0, 1].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
axes[0, 1].set_xlabel('Mean Predicted Probability')
axes[0, 1].set_ylabel('Fraction of Positives')
axes[0, 1].set_title('Calibration Plots', fontsize=14, fontweight='bold')
axes[0, 1].legend(loc='lower right')
axes[0, 1].grid(alpha=0.3)

# Brier score decomposition
brier_scores = {}
for name, y_pred in models_pred.items():
    brier_scores[name] = brier_score_loss(y_test, y_pred)

axes[1, 0].bar(range(len(brier_scores)), list(brier_scores.values()), 
              color=['blue', 'green', 'red'])
axes[1, 0].set_xticks(range(len(brier_scores)))
axes[1, 0].set_xticklabels(list(brier_scores.keys()))
axes[1, 0].set_ylabel('Brier Score (lower is better)')
axes[1, 0].set_title('Brier Score Comparison', fontsize=14, fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Expected Calibration Error (ECE)
def calculate_ece(y_true, y_prob, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_lowers = bin_boundaries[:-1]
    bin_uppers = bin_boundaries[1:]
    
    ece = 0
    for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
        in_bin = (y_prob > bin_lower) & (y_prob <= bin_upper)
        prop_in_bin = in_bin.mean()
        
        if prop_in_bin > 0:
            accuracy_in_bin = y_true[in_bin].mean()
            avg_confidence_in_bin = y_prob[in_bin].mean()
            ece += np.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    
    return ece

ece_scores = {}
for name, y_pred in models_pred.items():
    ece_scores[name] = calculate_ece(y_test, y_pred)

axes[1, 1].bar(range(len(ece_scores)), list(ece_scores.values()),
              color=['blue', 'green', 'red'])
axes[1, 1].set_xticks(range(len(ece_scores)))
axes[1, 1].set_xticklabels(list(ece_scores.keys()))
axes[1, 1].set_ylabel('Expected Calibration Error (lower is better)')
axes[1, 1].set_title('ECE Comparison', fontsize=14, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'pr_calibration_analysis.png'), dpi=300)
plt.show()

print("\nCalibration Metrics:")
for name in models_pred.keys():
    print(f"\n{name}:")
    print(f"  Brier Score: {brier_scores[name]:.4f}")
    print(f"  ECE: {ece_scores[name]:.4f}")

print(f"\nBest Calibrated Model: {min(brier_scores, key=brier_scores.get)}")

## 25. Early Warning System Analysis

In [ ]:
# Cell 25: Early warning analysis at different time points

print("\n" + "="*70)
print("EARLY WARNING SYSTEM ANALYSIS")
print("="*70)

if X_test_temporal is not None and lstm_model is not None:
    # Test at different weeks
    time_points = [4, 8, 12, 16, 20, 24, 28, 32]
    early_warning_results = []
    
    for week in time_points:
        # Use only first N weeks (zero-pad the rest)
        X_test_partial = np.zeros_like(X_test_temporal)
        X_test_partial[:, :week, :] = X_test_temporal[:, :week, :]
        
        # Get predictions
        partial_pred, _ = lstm_model.predict(X_test_partial, verbose=0)
        
        # Calculate metrics
        auc = roc_auc_score(y_test, partial_pred)
        f1 = f1_score(y_test, (partial_pred > 0.5).astype(int))
        accuracy = accuracy_score(y_test, (partial_pred > 0.5).astype(int))
        
        early_warning_results.append({
            'Week': week,
            'AUC': auc,
            'F1': f1,
            'Accuracy': accuracy
        })
    
    early_df = pd.DataFrame(early_warning_results)
    
    print("\nEarly Warning Performance:")
    print(early_df.round(3))
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # AUC over time
    axes[0].plot(early_df['Week'], early_df['AUC'], 'o-', color='blue', linewidth=2, markersize=8)
    axes[0].axhline(y=0.7, color='red', linestyle='--', alpha=0.5, label='Acceptable (0.7)')
    axes[0].axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Good (0.8)')
    axes[0].set_xlabel('Weeks of Data')
    axes[0].set_ylabel('AUC Score')
    axes[0].set_title('Prediction Quality vs Time', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # F1 over time
    axes[1].plot(early_df['Week'], early_df['F1'], 'o-', color='green', linewidth=2, markersize=8)
    axes[1].axhline(y=0.6, color='red', linestyle='--', alpha=0.5, label='Acceptable (0.6)')
    axes[1].set_xlabel('Weeks of Data')
    axes[1].set_ylabel('F1 Score')
    axes[1].set_title('F1 Score vs Time', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    # Improvement rate
    improvement = (early_df['AUC'].values[1:] - early_df['AUC'].values[:-1]) / early_df['AUC'].values[:-1] * 100
    axes[2].bar(early_df['Week'].values[1:], improvement, color='purple', alpha=0.7)
    axes[2].set_xlabel('Weeks of Data')
    axes[2].set_ylabel('% Improvement from Previous')
    axes[2].set_title('Marginal Improvement in AUC', fontsize=14, fontweight='bold')
    axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'early_warning_analysis.png'), dpi=300)
    plt.show()
    
    # Find minimum weeks for acceptable performance
    acceptable_weeks = early_df[early_df['AUC'] >= 0.7]['Week'].min()
    good_weeks = early_df[early_df['AUC'] >= 0.8]['Week'].min()
    
    print(f"\nEarly Warning Recommendations:")
    print(f"  ✓ Minimum weeks for acceptable performance (AUC ≥ 0.7): Week {acceptable_weeks}")
    if pd.notna(good_weeks):
        print(f"  ✓ Minimum weeks for good performance (AUC ≥ 0.8): Week {good_weeks}")
    print(f"  ✓ Performance plateaus after week {early_df[early_df['AUC'] == early_df['AUC'].max()]['Week'].min()}")
else:
    print("⚠️ Early warning analysis not available")
    early_df = pd.DataFrame()

## 26. Comprehensive JSON Export with All Metrics

In [ ]:
# Cell 26: Update JSON with all comprehensive metrics

print("\n" + "="*70)
print("EXPORTING COMPREHENSIVE RESULTS")
print("="*70)

# Prepare comprehensive results JSON
comprehensive_results = {
    "timestamp": datetime.now().isoformat(),
    "sequence_length_weeks": 32,
    "models": {
        "lstm": {
            "auc": float(lstm_auc_test),
            "f1": float(lstm_f1_test),
            "accuracy": float(lstm_acc_test),
            "brier_score": float(brier_scores.get('LSTM', 0)),
            "ece": float(ece_scores.get('LSTM', 0))
        },
        "rf": {
            "auc": float(rf_auc_test),
            "f1": float(rf_f1_test),
            "accuracy": float(rf_acc_test),
            "brier_score": float(brier_scores.get('Random Forest', 0)),
            "ece": float(ece_scores.get('Random Forest', 0))
        },
        "hybrid_lstm_rf": {
            "auc": float(hybrid_auc),
            "f1": float(hybrid_f1),
            "accuracy": float(hybrid_acc),
            "method": hybrid_method,
            "blend_alpha": float(optimal_alpha),
            "brier_score": float(brier_scores.get('Hybrid', 0)),
            "ece": float(ece_scores.get('Hybrid', 0))
        }
    },
    "confusion_matrix": cm_dict,
    "threshold_analysis": {
        "optimal_f1_threshold": float(optimal_f1_threshold),
        "balanced_threshold": float(balanced_threshold),
        "optimal_f1_score": float(threshold_df.loc[optimal_f1_idx, 'F1']),
        "intervention_rate": float(threshold_df.loc[optimal_f1_idx, 'Intervention_Rate'])
    },
    "calibration": {
        "brier": float(brier_scores.get('Hybrid', 0)),
        "ece": float(ece_scores.get('Hybrid', 0))
    },
    "by_institution": institution_metrics.to_dict('records') if len(institution_metrics) > 0 else [],
    "by_language": language_df.to_dict('records') if len(language_df) > 0 else [],
    "generalizability": {
        "auc_variance": float(institution_metrics['AUC'].var()) if len(institution_metrics) > 0 else 0,
        "auc_cv": float(institution_metrics['AUC'].std() / institution_metrics['AUC'].mean()) if len(institution_metrics) > 0 else 0
    },
    "fairness_gaps": {
        "language_tpr_gap": float(language_df['TPR'].max() - language_df['TPR'].min()) if len(language_df) > 0 else 0,
        "language_fpr_gap": float(language_df['FPR'].max() - language_df['FPR'].min()) if len(language_df) > 0 else 0,
        "language_auc_gap": float(language_df['AUC'].max() - language_df['AUC'].min()) if len(language_df) > 0 else 0
    },
    "early_warning": early_df.to_dict('records') if len(early_df) > 0 else [],
    "critical_weeks": critical_week_numbers[:5],
    "feature_importance_hint": top_features[:10] if top_features else [],
    "dataset_info": {
        "train_samples": len(train_df),
        "val_samples": len(val_df),
        "test_samples": len(test_df),
        "positive_class_ratio": float(y_train.mean())
    },
    "notes": "Comprehensive analysis including cultural distance, teaching style mismatch, support programs, and fairness metrics."
}

# Save comprehensive JSON
json_path = os.path.join(RESULTS_DIR, 'model_results_32w.json')
with open(json_path, 'w') as f:
    json.dump(comprehensive_results, f, indent=2)

print(f"✓ Saved comprehensive results to {json_path}")

# Display summary of what was saved
print("\nExported Metrics Summary:")
for key in comprehensive_results.keys():
    if isinstance(comprehensive_results[key], dict):
        print(f"  - {key}: {len(comprehensive_results[key])} items")
    elif isinstance(comprehensive_results[key], list):
        print(f"  - {key}: {len(comprehensive_results[key])} records")
    else:
        print(f"  - {key}")

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)
print("\n✅ All advanced analyses completed successfully!")
print("✅ Results exported for dashboard visualization")
print("✅ All visualizations saved to plots directory")